# Extract data from Customer_JSON
### - Query Single Files
### - Query list of files using wildcard Characters
### - Query all files in floders
### - Write to bronze layer



# 1.Query Single File


In [0]:
%fs ls /Volumes/gizmobox_nara/landing/operational_volume/customers

In [0]:
%sql
SELECT * FROM json.`dbfs:/Volumes/gizmobox_nara/landing/operational_volume/customers/customers_2024_10.json`

In [0]:
 df = spark.read.format("json").load("/Volumes/gizmobox_nara/landing/operational_volume/customers/customers_2024_10.json")
 display(df)


# 2.Query list of Multiple Files

In [0]:
df= spark.read.format("json").load("/Volumes/gizmobox_nara/landing/operational_volume/customers/customers_2024_*.json")
display(df)

In [0]:
%sql
SELECT * FROM json.`dbfs:/Volumes/gizmobox_nara/landing/operational_volume/customers/customers_2024_*.json`

# 3. Query all files in Floder

In [0]:
df = spark.read.json("/Volumes/gizmobox_nara/landing/operational_volume/customers")
display(df)

In [0]:
%sql
SELECT * FROM json.`dbfs:/Volumes/gizmobox_nara/landing/operational_volume/customers`

--Here are the DataFrames in the session:
--- Spark DataFrame

# SELECT FILE METADATA

In [0]:
from pyspark.sql.functions import col
df_with_metadata = df.select(col("_metadata.file_path"), "*")
display(df_with_metadata)

In [0]:
%sql
SELECT input_file_name() As filepath,
_metadata.file_path As filepath
from json.`dbfs:/Volumes/gizmobox_nara/landing/operational_volume/customers/`

In [0]:
%sql
CREATE OR REPLACE VIEW gizmobox_nara.bronze.v_customers AS
SELECT *, _metadata.file_path AS filepath
FROM json.`dbfs:/Volumes/gizmobox_nara/landing/operational_volume/customers/`

In [0]:
df_with_metadata.write.format("delta").model("overwrite").saveAsTable("gizmobox_nara.bronze.py_customers")

In [0]:
df_with_metadata.writeTo("gizmobox_nara.bronze.py_customers").createOrReplace()

In [0]:
df = spark.table("gizmobox_nara.bronze.py_customers")
display(df)

In [0]:
%sql
SELECT * FROM gizmobox_nara.bronze.py_customers

# CREATING TEMPORARY AND GLOBAL VIEWS